# 06 — Robustness Experiments: Training Seeds and Letterbox Padding
**Project:** From Prediction to Referral — Uncertainty-Aware Pneumonia Detection under Cross-Population Distribution Shift

This notebook adds two **pre-specified robustness analyses** to the locked study. It does not change any primary result:
the primary analysis remains the locked Notebook 02 models (seed 42, letterbox preprocessing).

**Pre-specified before any output of this notebook existed (2026-10-28):**
| Item | Choice |
|---|---|
| R1 — training seed | retrain both architectures with seeds **43 and 44**, identical recipe (letterbox). Together with the locked seed-42 models: 3 runs per architecture |
| R2 — letterbox padding | retrain both architectures with seed 42 using **direct resize without padding ("stretch")**; everything else identical |
| Per-run operating decisions | threshold A (max validation sensitivity s.t. specificity ≥ 0.95), temperature, referral threshold τ (90th percentile of validation MC entropy) — all re-derived on **Mooney validation of that run only** |
| Locked seed-42 models | re-evaluated with the same code; their locked thresholds and temperatures are used, and the recomputed values are printed as a check |
| Cohorts | Mooney validation (fitting only), Mooney ID test, NIH AP / PA / AP-Hard adult (frozen evaluation) |
| Metrics | AUROC, AUPRC, specificity/sensitivity at A, % of NIH negatives called pneumonia, ECE (TS), MC-entropy error-detection AUROC, OOD-detection AUROC, error rate without referral vs selective risk at τ90, % of errors that are confidently incorrect (G4) |
| Statistics | patient-level bootstrap (B = 1000), **same draws for every run** (paired differences: stretch − letterbox seed 42) |
| R1 decision rule | a primary conclusion is "seed-robust" if it holds in all three runs of both architectures |
| R2 decision rule | the padding shortcut is supported if, without padding, (a) ID-test specificity at A rises and (b) the false-positive rate among ID-test normals no longer rises steeply with aspect ratio (Q4 − Q1 difference shrinks). NIH images are square, so their input is identical under both preprocessings; NIH effects of R2 are reported descriptively |

**Kaggle settings:** GPU T4 ×1, **Internet ON** (ImageNet weights). Use **Save Version → Save & Run All (Commit)** so it runs in the
background (≈ 3–5 h). Inputs: `paultimothymooney/chest-xray-pneumonia`, `nih-chest-xrays/data`, your `cxr-ood-notebook01-output`
dataset, the **01b v3** output (locked cohorts) and the **locked Notebook 02** output (a guard checks the version).

In [ ]:
# ------------------------------------------------------------------ CONFIG
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
NIH_ROOT   = Path("/kaggle/input/datasets/nih-chest-xrays/data")
PROJECT    = Path("/kaggle/working/CXR_OOD_Project")
MODELS     = ["efficientnet_b4", "vit_b16"]
NEW_RUNS   = [("lb", 43), ("lb", 44), ("st", 42)]     # (preprocessing, seed) trained here; ("lb", 42) = locked Notebook 02 models
T_MC       = 50
N_BOOT     = 1000
BOOT_SEED  = 20261028
ECE_BINS   = 15
BATCH      = 32
SENS_SPEC  = 0.95
SMOKE_TEST = False
PRETRAINED = True

MODEL_CFG = {
    "efficientnet_b4": dict(timm_name="efficientnet_b4.ra2_in1k", img_size=384, batch_size=16, accum=2,
                            lr_backbone=1e-4, lr_head=1e-3, weight_decay=1e-2, drop_path=0.1),
    "vit_b16":         dict(timm_name="vit_base_patch16_224.augreg_in21k_ft_in1k", img_size=224, batch_size=32, accum=1,
                            lr_backbone=3e-5, lr_head=1e-3, weight_decay=5e-2, drop_path=0.1),
}
COMMON = dict(max_epochs=30, min_epochs=5, patience=6, warmup_epochs=1, grad_clip=1.0, head_dropout=0.3, head_hidden=256)
LOCKED = {  # Notebook 02 version A (2026-10-05 08:09 UTC): threshold A (logit) and temperature
    "efficientnet_b4": dict(thrA=0.4689939068127956, T=1.5296217972658805),
    "vit_b16":         dict(thrA=-0.28979498040711366, T=1.4168967422767194),
}
EXPECTED_VAL_PRED_SHA = {"efficientnet_b4": "33a07785b024d7408591941c4461bd314a9ff63988628709b90efcb7e12ea5fd",
                         "vit_b16": "cc21299bc2cd00b7555eec65ab34f61df2488d59cd78ebaba678e8d8cc58c43f"}
EXPECTED_FP = {"MOONEY_VAL": "13d0556c2de25782967c5cd20ceb54508ca4b8061cdca9e0c9c6ae0bff2f139c",
               "MOONEY_TEST": "723c6c2e962c6704b4e61e693ea96f831104cd9f937acbbf309b46b0edc55101",
               "MOONEY_TRAIN": "762c050910f4560893544f55ef702ba22d80dd948d0a9b007a57b9f49ce79f4b",
               "OOD_AP_ADULT": "b13bc8c6413b3a5725e08184ade4a34305288d0daff717533589399c83e0d567",
               "OOD_PA_ADULT": "ede8eeb671334525f5d4d0edce8a8a025bcf7992bb19d9060e8a17c6bd116cb0",
               "OOD_HARD_AP_ADULT": "b8cc02779f56de273cd8ffcca8bc5915451e279f97f9b42f2211c1f1c544fb60"}
ID, OOD = "MOONEY_TEST", ["OOD_AP_ADULT", "OOD_PA_ADULT", "OOD_HARD_AP_ADULT"]
EVAL = [ID] + OOD

In [ ]:
import os, re, sys, json, time, math, random, hashlib, shutil, platform, datetime
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np, pandas as pd
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as TV
from PIL import Image
from scipy.optimize import minimize_scalar
from scipy.special import xlogy, expit
from concurrent.futures import ThreadPoolExecutor
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

DIRS = {k: PROJECT / k for k in ["checkpoints", "predictions", "metrics", "tables", "figures", "logs", "code", "cache"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise SystemExit("No GPU detected — set Accelerator to GPU T4 and restart.")

PRUNE = re.compile(r"^(images(_\d+)?|train|test|val|NORMAL|PNEUMONIA|chest_xray|__MACOSX)$")
def find_one(name):
    hits = []
    for root, dirs, files in os.walk(INPUT_ROOT):
        dirs[:] = [d for d in dirs if not PRUNE.match(d)]
        if name in files: hits.append(Path(root) / name)
    if not hits: raise SystemExit(f"'{name}' not found under {INPUT_ROOT}. Check Add Input.")
    if len({hashlib.sha256(h.read_bytes()).hexdigest() for h in hits}) > 1:
        raise SystemExit(f"Found DIFFERENT copies of '{name}': {hits}. Remove the wrong input.")
    return sorted(hits)[0]

FILES = {n: find_one(n) for n in ["mooney_manifest_final.csv", "cxr_common.py"] + [f"{m}_best.pt" for m in MODELS] +
         [f"{m}_val_predictions.csv" for m in MODELS]}
def find_optional(name):
    try: return find_one(name)
    except SystemExit: return None
for c in OOD: FILES[f"{c}.csv"] = find_optional(f"{c}.csv")
for m, h in EXPECTED_VAL_PRED_SHA.items():
    ok = hashlib.sha256(FILES[f"{m}_val_predictions.csv"].read_bytes()).hexdigest() == h
    print(f"[{'PASS' if ok else 'FAIL'}] Notebook 02 version check ({m})")
    if not ok and not SMOKE_TEST: raise SystemExit("Wrong Notebook 02 version attached — use the locked version (2026-10-05T08:09).")
shutil.copy(FILES["cxr_common.py"], DIRS["code"] / "cxr_common.py"); sys.path.insert(0, str(DIRS["code"]))
import importlib, cxr_common; importlib.reload(cxr_common)
from cxr_common import load_letterboxed, to_model_tensor, CXRClassifier

ENV = dict(python=platform.python_version(), torch=torch.__version__, timm=timm.__version__, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
           timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(DIRS["logs"] / "environment_06.json").write_text(json.dumps(ENV, indent=2)); print(ENV)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

## 1. Cohorts (verified against the locked fingerprints)

In [ ]:
_MROOT = []
def resolve(p):
    """Manifest paths are keys; if the Mooney dataset is mounted elsewhere, map them to the actual location."""
    if os.path.exists(p) or "chest_xray/" not in p: return p
    if not _MROOT:
        cands = []
        for root, dirs, files in os.walk(INPUT_ROOT):
            dirs[:] = [d for d in dirs if d not in ("NORMAL", "PNEUMONIA", "__MACOSX") and not d.startswith("images")]
            if os.path.basename(root) == "chest_xray" and os.path.isdir(os.path.join(root, "train", "NORMAL")) and os.path.isdir(os.path.join(root, "test")):
                cands.append(root)
        if not cands: raise SystemExit("Mooney 'chest_xray' folder not found under /kaggle/input — add the dataset paultimothymooney/chest-xray-pneumonia via Add Input.")
        _MROOT.append(min(cands, key=len)); print("Mooney images found at:", _MROOT[0])
    q = os.path.join(_MROOT[0], p.split("chest_xray/", 1)[1])
    if not os.path.exists(q): raise SystemExit(f"Mooney image missing: {q}")
    return q

def fingerprint(paths): return hashlib.sha256("\n".join(sorted(paths)).encode()).hexdigest()
man = pd.read_csv(FILES["mooney_manifest_final.csv"])
COH = {}
for name, split in [("MOONEY_TRAIN", "train"), ("MOONEY_VAL", "val"), ("MOONEY_TEST", "test_id")]:
    d = man[man.final_split == split].sort_values("filepath").reset_index(drop=True)
    COH[name] = pd.DataFrame({"cohort": name, "filepath": d.filepath, "patient_id": "M:" + d.patient_id.astype(str), "y": d.y.astype(int)})
    if {"width", "height"} <= set(d.columns): COH[name]["aspect"] = (d.width / d.height).values
del man
nih_index = {p.name: str(p) for p in NIH_ROOT.glob("images_*/images/*.png")}
def rebuild_nih_cohort(c):
    """Fallback when the 01b output is not attached: rebuild the cohort from Data_Entry_2017.csv with the locked Phase 1 rules.
    The SHA-256 fingerprint check below guarantees the result is identical to the locked cohort."""
    meta = pd.read_csv(NIH_ROOT / "Data_Entry_2017.csv")
    labels = meta["Finding Labels"].astype(str).str.split("|").apply(set)
    age = pd.to_numeric(meta["Patient Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    pos = labels.apply(lambda L: "Pneumonia" in L)
    neg = (labels.apply(lambda L: "Pneumonia" not in L and bool(L & {"Consolidation", "Infiltration", "Edema", "Atelectasis", "Effusion"}))
           if c == "OOD_HARD_AP_ADULT" else labels.apply(lambda L: L == {"No Finding"}))
    view = "PA" if c == "OOD_PA_ADULT" else "AP"
    keep = (meta["View Position"] == view) & (pos | neg) & age.between(18, 100)
    return pd.DataFrame({"filename": meta.loc[keep, "Image Index"].values, "patient_id": meta.loc[keep, "Patient ID"].values,
                         "y": pos[keep].astype(int).values})
for c in OOD:
    if FILES[f"{c}.csv"] is not None: d = pd.read_csv(FILES[f"{c}.csv"])
    else: print(f"{c}.csv not attached — rebuilding from Data_Entry_2017.csv"); d = rebuild_nih_cohort(c)
    d = d.sort_values("filename").reset_index(drop=True)
    if (~d.filename.isin(nih_index)).sum(): raise SystemExit(f"{c}: images missing under NIH_ROOT")
    COH[c] = pd.DataFrame({"cohort": c, "filepath": d.filename.map(nih_index), "patient_id": "N:" + d.patient_id.astype(str),
                           "y": d.y.astype(int), "filename": d.filename})
checks = [(c, fingerprint(COH[c].filepath) == EXPECTED_FP[c]) for c in ["MOONEY_TRAIN", "MOONEY_VAL", "MOONEY_TEST"]]
checks += [(c, hashlib.sha256("\n".join(COH[c].filename).encode()).hexdigest() == EXPECTED_FP[c]) for c in OOD]
for c, ok in checks: print(f"[{'PASS' if ok else 'FAIL'}] cohort {c}: n={len(COH[c])}")
if not all(ok for _, ok in checks) and not SMOKE_TEST: raise SystemExit("Cohort verification failed.")
if SMOKE_TEST:
    COH = {k: v.groupby("y", group_keys=False).head(12).reset_index(drop=True) for k, v in COH.items()}
    COMMON.update(max_epochs=1, min_epochs=1, patience=1); T_MC, N_BOOT = 5, 20
for k in COH:   # aspect ratio of Mooney images (read from file if the manifest lacks width/height)
    if k.startswith("MOONEY") and "aspect" not in COH[k]:
        COH[k]["aspect"] = [(lambda s: s[0] / s[1])(Image.open(resolve(p)).size) for p in COH[k].filepath]

## 2. Preprocessing: letterbox (locked) and stretch (R2)

In [ ]:
def load_pre(path, sizes):
    """Decode once; return {('lb', z): letterbox, ('st', z): stretch}. For square images both are identical."""
    im = Image.open(resolve(path)).convert("L"); w, h = im.size; s = max(w, h)
    out = {}
    canvas = Image.new("L", (s, s), 0); canvas.paste(im, ((s - w) // 2, (s - h) // 2))
    for z in sizes:
        out[("lb", z)] = np.array(canvas.resize((z, z), Image.LANCZOS), dtype=np.uint8)
        out[("st", z)] = out[("lb", z)] if w == h else np.array(im.resize((z, z), Image.LANCZOS), dtype=np.uint8)
    return out
for p in list(COH["MOONEY_VAL"].filepath[:2]) + list(COH["OOD_AP_ADULT"].filepath[:1]):
    for z in {MODEL_CFG[m]["img_size"] for m in MODELS}:
        assert np.array_equal(load_pre(p, [z])[("lb", z)], load_letterboxed(resolve(p), z)), "letterbox differs from Notebook 02"
print("letterbox identical to Notebook 02")

CACHE = {}
def get_cache(pre, size, split):
    key = (pre, size, split)
    if key not in CACHE:
        with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
            CACHE[key] = np.stack(list(ex.map(lambda p: load_pre(p, [size])[(pre, size)], COH[split].filepath)))
        print(f"cached {key}: {CACHE[key].shape}")
    return CACHE[key]

## 3. Training (identical recipe to Notebook 02; only seed or preprocessing differs)

In [ ]:
class TrainDS(Dataset):
    def __init__(self, imgs, labels, train):
        self.imgs, self.labels, self.train = imgs, labels.astype(np.float32), train
        self.aug = TV.Compose([TV.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05), fill=0),
                               TV.ColorJitter(brightness=0.1, contrast=0.1)])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).unsqueeze(0)
        return (self.aug(x) if self.train else x), torch.tensor(self.labels[i])
def seed_worker(wid): s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

@torch.no_grad()
def predict(model, loader):
    model.eval(); L = []
    for x, _ in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            L.append(model(to_model_tensor(x.to(DEVICE))).float().cpu())
    return torch.cat(L).numpy()

def run_id(m, pre, seed): return f"{m}__{pre}_s{seed}"

def train_run(m, pre, seed):
    rid = run_id(m, pre, seed); ckpt = DIRS["checkpoints"] / f"{rid}.pt"
    if ckpt.exists(): print(f"{rid}: checkpoint exists — skipped"); return
    cfg = {**MODEL_CFG[m], **COMMON}; seed_everything(seed)
    tr, va = get_cache(pre, cfg["img_size"], "MOONEY_TRAIN"), get_cache(pre, cfg["img_size"], "MOONEY_VAL")
    g = torch.Generator(); g.manual_seed(seed); nw = min(4, os.cpu_count())
    dl_tr = DataLoader(TrainDS(tr, COH["MOONEY_TRAIN"].y.values, True), batch_size=cfg["batch_size"], shuffle=True, num_workers=nw,
                       worker_init_fn=seed_worker, generator=g, pin_memory=True, drop_last=True, persistent_workers=nw > 0)
    dl_va = DataLoader(TrainDS(va, COH["MOONEY_VAL"].y.values, False), batch_size=cfg["batch_size"] * 2, num_workers=nw)
    model = CXRClassifier(cfg["timm_name"], pretrained=PRETRAINED, drop_path=cfg["drop_path"],
                          head_dropout=cfg["head_dropout"], head_hidden=cfg["head_hidden"]).to(DEVICE)
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": cfg["lr_backbone"]}, {"params": head, "lr": cfg["lr_head"]}],
                            weight_decay=cfg["weight_decay"])
    spe = max(1, len(dl_tr) // cfg["accum"]); total, warm = spe * cfg["max_epochs"], spe * cfg["warmup_epochs"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda"); crit = nn.BCEWithLogitsLoss()
    yv = COH["MOONEY_VAL"].y.values.astype(np.float32); hist, best, bad = [], float("inf"), 0
    for ep in range(1, cfg["max_epochs"] + 1):
        model.train(); t0 = time.time(); opt.zero_grad(set_to_none=True)
        for i, (x, y) in enumerate(dl_tr):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = crit(model(to_model_tensor(x.to(DEVICE))).float(), y.to(DEVICE)) / cfg["accum"]
            scaler.scale(loss).backward()
            if (i + 1) % cfg["accum"] == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        lv = predict(model, dl_va); vloss = float(crit(torch.tensor(lv), torch.tensor(yv)))
        hist.append(dict(epoch=ep, val_loss=vloss, seconds=time.time() - t0))
        if vloss < best - 1e-4:
            best, bad = vloss, 0
            torch.save({"model_state": model.state_dict(), "config": cfg, "epoch": ep, "val_loss": vloss, "seed": seed, "pre": pre, "env": ENV}, ckpt)
        else: bad += 1
        print(f"[{rid}] ep {ep:02d} val loss {vloss:.4f} ({hist[-1]['seconds']:.0f}s){' *' if bad == 0 else ''}")
        if ep >= cfg["min_epochs"] and bad >= cfg["patience"]: break
    pd.DataFrame(hist).to_csv(DIRS["logs"] / f"{rid}_history.csv", index=False)
    del model; torch.cuda.empty_cache()

for pre, seed in NEW_RUNS:
    for m in MODELS: train_run(m, pre, seed)
for k in [k for k in CACHE if k[2] == "MOONEY_TRAIN"]: del CACHE[k]

## 4. Inference: deterministic logits + 50 MC-Dropout passes (head only) for every run, every image decoded once

In [ ]:
RUNS = {}   # rid -> dict(model, pre, seed, ckpt)
for m in MODELS:
    RUNS[run_id(m, "lb", 42)] = dict(model=m, pre="lb", seed=42, ckpt=FILES[f"{m}_best.pt"], locked=True)
    for pre, seed in NEW_RUNS:
        RUNS[run_id(m, pre, seed)] = dict(model=m, pre=pre, seed=seed, ckpt=DIRS["checkpoints"] / f"{run_id(m, pre, seed)}.pt", locked=False)
NETS = {}
for rid, r in RUNS.items():
    st = torch.load(r["ckpt"], map_location="cpu", weights_only=False); cfg = st["config"]
    net = CXRClassifier(cfg["timm_name"], pretrained=False, drop_path=cfg["drop_path"], head_dropout=cfg["head_dropout"], head_hidden=cfg["head_hidden"])
    net.load_state_dict(st["model_state"]); NETS[rid] = net.to(DEVICE).eval(); r["size"] = cfg["img_size"]; r["epoch"] = st["epoch"]
    print(f"{rid}: best epoch {st['epoch']}, val loss {st['val_loss']:.4f}")
SIZES = sorted({r["size"] for r in RUNS.values()})

class Imgs(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        a = load_pre(self.paths[i], SIZES)
        return {f"{p}{z}": torch.from_numpy(v).unsqueeze(0) for (p, z), v in a.items()}, i

@torch.no_grad()
def infer(paths, chunk_id):
    dl = DataLoader(Imgs(paths), batch_size=BATCH, num_workers=min(4, os.cpu_count()), pin_memory=True)
    det = {r: np.empty(len(paths), np.float32) for r in RUNS}; mc = {r: np.empty((len(paths), T_MC), np.float16) for r in RUNS}
    t0, n = time.time(), 0
    for b, (xs, idx) in enumerate(dl):
        for j, (rid, net) in enumerate(NETS.items()):
            r = RUNS[rid]; x = to_model_tensor(xs[f"{r['pre']}{r['size']}"].to(DEVICE, non_blocking=True))
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                f = net.features(x)
            f = f.float(); det[rid][idx.numpy()] = net.head(f).cpu().numpy()
            for d_ in net.mc_dropout_modules(): d_.train()
            torch.manual_seed(BOOT_SEED + chunk_id * 100_003 + b * 101 + j)
            mc[rid][idx.numpy()] = torch.stack([net.head(f) for _ in range(T_MC)], 1).cpu().numpy().astype(np.float16)
            for d_ in net.mc_dropout_modules(): d_.eval()
        n += len(idx)
        if b % 100 == 0: print(f"  chunk {chunk_id}: {n}/{len(paths)} ({n / (time.time() - t0):.0f} img/s)")
    return det, mc

ALL = sorted(set(pd.concat([COH[c].filepath for c in ["MOONEY_VAL"] + EVAL])))
DET = {r: np.empty(len(ALL), np.float32) for r in RUNS}; MC = {r: np.empty((len(ALL), T_MC), np.float16) for r in RUNS}
CH = 10000; tag = hashlib.md5(json.dumps(sorted(RUNS)).encode()).hexdigest()[:8]
for k, i in enumerate(range(0, len(ALL), CH)):
    f = DIRS["cache"] / f"chunk_{tag}_{k:03d}.npz"; part = ALL[i:i + CH]
    if f.exists() and list(np.load(f, allow_pickle=True)["paths"]) == part:
        z = np.load(f, allow_pickle=True); d, mcs = {r: z[f"d_{r}"] for r in RUNS}, {r: z[f"m_{r}"] for r in RUNS}
    else:
        d, mcs = infer(part, k)
        np.savez(f, paths=np.array(part, dtype=object), **{f"d_{r}": d[r] for r in RUNS}, **{f"m_{r}": mcs[r] for r in RUNS})
    for r in RUNS: DET[r][i:i + CH] = d[r]; MC[r][i:i + CH] = mcs[r]
POS = {p: i for i, p in enumerate(ALL)}

# reproduction check for the locked models
for m in MODELS:
    rid = run_id(m, "lb", 42); ref = pd.read_csv(FILES[f"{m}_val_predictions.csv"]).set_index("filepath")["logit"]
    v = COH["MOONEY_VAL"].filepath; diff = np.abs(DET[rid][[POS[p] for p in v]] - ref.loc[v].values)
    print(f"{rid}: validation max |Δlogit| vs Notebook 02 = {diff.max():.4f}")
    if diff.max() > 0.05 and not SMOKE_TEST: raise SystemExit("Locked model not reproduced — stop and report.")

## 5. Per-run operating decisions (validation only), per-image table

In [ ]:
def H2(p, q): return -(xlogy(p, p) + xlogy(q, q)) / np.log(2)
def thr_A(z, y):
    best = None
    for t in np.r_[np.unique(z), z.max() + 1e-6]:
        pr = z >= t; sens = (pr & (y == 1)).sum() / (y == 1).sum(); spec = (~pr & (y == 0)).sum() / (y == 0).sum()
        if spec >= SENS_SPEC:
            key = (sens, spec, t)
            if best is None or key > best: best = key
    return float(best[2])
def fit_T(z, y):
    nll = lambda lt: float(np.mean(np.logaddexp(0, z / np.exp(lt)) - y * z / np.exp(lt)))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

rows, OPS = [], {}
for rid, r in RUNS.items():
    zmc = MC[rid].astype(np.float64); p, q = expit(zmc), expit(-zmc)
    ent = H2(p.mean(1), q.mean(1)); zd = DET[rid].astype(np.float64); entd = H2(expit(zd), expit(-zd))
    iv = [POS[x] for x in COH["MOONEY_VAL"].filepath]; yv = COH["MOONEY_VAL"].y.values
    tA, T = thr_A(zd[iv], yv), fit_T(zd[iv], yv)
    if r["locked"]:
        print(f"{rid}: recomputed thrA {tA:.4f} vs locked {LOCKED[r['model']]['thrA']:.4f}; T {T:.4f} vs locked {LOCKED[r['model']]['T']:.4f}")
        tA, T = LOCKED[r["model"]]["thrA"], LOCKED[r["model"]]["T"]
    tau = float(np.quantile(ent[iv], 0.90, method="higher"))
    OPS[rid] = dict(**{k: r[k] for k in ["model", "pre", "seed", "epoch"]}, thrA=tA, T=T, tau90=tau)
    for c in ["MOONEY_VAL"] + EVAL:
        d = COH[c]; ix = [POS[x] for x in d.filepath]; pred = (zd[ix] >= tA).astype(int)
        rows.append(pd.DataFrame({"run": rid, "cohort": c, "filepath": d.filepath.values, "patient_id": d.patient_id.values, "y": d.y.values,
                                  "aspect": d["aspect"].values if "aspect" in d else np.nan, "logit": zd[ix], "prob_ts": expit(zd[ix] / T),
                                  "pred": pred, "error": (pred != d.y.values).astype(int), "entropy_mc": ent[ix], "entropy_det": entd[ix],
                                  "accepted": (ent[ix] <= tau).astype(int)}))
P = pd.concat(rows, ignore_index=True)
P.to_csv(DIRS["predictions"] / "robustness_per_image.csv.gz", index=False)
pd.DataFrame(OPS).T.to_csv(DIRS["tables"] / "robustness_operating_points.csv")
print(pd.DataFrame(OPS).T)

## 6. Metrics with patient-level bootstrap (identical draws for every run)

In [ ]:
def ranks(s):
    u, inv = np.unique(s, return_inverse=True); return inv, len(u)
def w_auc(rk, lab, w):
    inv, K = rk; pos = np.bincount(inv, w * lab, K); neg = np.bincount(inv, w * (1 - lab), K); Pn, N = pos.sum(), neg.sum()
    return float(np.sum(pos * (np.cumsum(neg) - 0.5 * neg)) / (Pn * N)) if Pn > 0 and N > 0 else np.nan
def w_ap(rk, lab, w):
    inv, K = rk; pos = np.bincount(inv, w * lab, K)[::-1]; neg = np.bincount(inv, w * (1 - lab), K)[::-1]
    tp, fp = np.cumsum(pos), np.cumsum(neg)
    return float(np.sum(pos / tp[-1] * tp / np.maximum(tp + fp, 1e-12)))
def ece(p, y, w):
    b = np.minimum((p * ECE_BINS).astype(int), ECE_BINS - 1)
    return float(np.abs(np.bincount(b, w * (y - p), ECE_BINS)).sum() / w.sum())
def prep(d, dI=None):
    k = dict(lg=ranks(d.logit.values), em=ranks(d.entropy_mc.values), ed=ranks(d.entropy_det.values))
    if dI is not None: k["oo"] = ranks(np.r_[dI.entropy_mc.values, d.entropy_mc.values]); k["oolab"] = np.r_[np.zeros(len(dI)), np.ones(len(d))]
    return k
def metrics(d, k, w, wI=None):
    y, e, a, pr = d.y.values, d.error.values, d.accepted.values, d.pred.values; sd = lambda x, z: x / z if z > 0 else np.nan
    out = dict(auroc=w_auc(k["lg"], y, w), auprc=w_ap(k["lg"], y, w),
               sens_A=sd((w * (pr == 1) * (y == 1)).sum(), (w * (y == 1)).sum()),
               spec_A=sd((w * (pr == 0) * (y == 0)).sum(), (w * (y == 0)).sum()),
               ece_ts=ece(d.prob_ts.values, y, w), errdet_auroc=w_auc(k["em"], e, w),
               errdet_auroc_det=w_auc(k["ed"], e, w), error_rate=sd((w * e).sum(), w.sum()),
               coverage=sd((w * a).sum(), w.sum()), selective_risk=sd((w * a * e).sum(), (w * a).sum()),
               g4_pct_of_errors=100 * sd((w * a * e).sum(), (w * e).sum()))
    out["risk_change"] = out["selective_risk"] - out["error_rate"]
    if "oo" in k: out["ooddet_auroc"] = w_auc(k["oo"], k["oolab"], np.r_[wI, w])
    return out

rng = np.random.default_rng(BOOT_SEED); DRAWS = {}
for c in EVAL:
    codes, u = pd.factorize(COH[c].patient_id.values)
    DRAWS[c] = (codes, [np.bincount(rng.integers(0, len(u), len(u)), minlength=len(u)) for _ in range(N_BOOT)])
G = {k: g.reset_index(drop=True) for k, g in P.groupby(["run", "cohort"])}
for k, g in G.items(): assert (g.filepath.values == COH[k[1]].filepath.values).all()

res, BOOT = [], {}
t0 = time.time()
for rid in RUNS:
    dI = G[(rid, ID)]; cI, drI = DRAWS[ID]
    for c in EVAL:
        d = G[(rid, c)]; codes, dr = DRAWS[c]; isood = c != ID
        kk = prep(d, dI if isood else None)
        pt = metrics(d, kk, np.ones(len(d)), np.ones(len(dI)) if isood else None)
        bs = [metrics(d, kk, cnt[codes].astype(float), drI[b][cI].astype(float) if isood else None) for b, cnt in enumerate(dr)]
        B = pd.DataFrame(bs); BOOT[(rid, c)] = B
        for k, v in pt.items():
            lo, hi = np.nanpercentile(B[k], [2.5, 97.5])
            res.append(dict(run=rid, **{x: OPS[rid][x] for x in ["model", "pre", "seed"]}, cohort=c, metric=k, estimate=v, ci_low=lo, ci_high=hi))
    print(f"{rid}: bootstrap done ({time.time() - t0:.0f}s)")
R = pd.DataFrame(res); R.to_csv(DIRS["metrics"] / "robustness_metrics_with_ci_long.csv", index=False)

## 7. R1 — seed variability and seed-robustness of the primary conclusions

In [ ]:
lb = R[R.pre == "lb"]
SEEDV = lb.groupby(["model", "cohort", "metric"]).estimate.agg(["mean", "std", "min", "max", "count"]).reset_index()
SEEDV.to_csv(DIRS["tables"] / "R1_seed_variability.csv", index=False)
def est(rid, c, k): return R[(R.run == rid) & (R.cohort == c) & (R.metric == k)].iloc[0]
checks = []
for m in MODELS:
    for s in [42] + [sd for p_, sd in NEW_RUNS if p_ == "lb"]:
        rid = run_id(m, "lb", s)
        for c in OOD:   # ΔAUROC (OOD − ID) with independent draws
            dlt = BOOT[(rid, c)].auroc.values - BOOT[(rid, ID)].auroc.values
            checks.append(dict(run=rid, conclusion=f"AUROC drop on {c}", value=est(rid, c, "auroc").estimate - est(rid, ID, "auroc").estimate,
                               ci_low=np.nanpercentile(dlt, 2.5), ci_high=np.nanpercentile(dlt, 97.5), holds=bool(np.nanpercentile(dlt, 97.5) < 0)))
        checks.append(dict(run=rid, conclusion="ID errdet AUROC > 0.5", value=est(rid, ID, "errdet_auroc").estimate,
                           ci_low=est(rid, ID, "errdet_auroc").ci_low, ci_high=est(rid, ID, "errdet_auroc").ci_high, holds=bool(est(rid, ID, "errdet_auroc").ci_low > 0.5)))
        for c in ["OOD_AP_ADULT", "OOD_HARD_AP_ADULT"]:
            e_ = est(rid, c, "errdet_auroc")
            checks.append(dict(run=rid, conclusion=f"errdet AUROC < 0.5 on {c}", value=e_.estimate, ci_low=e_.ci_low, ci_high=e_.ci_high, holds=bool(e_.ci_high < 0.5)))
        for c in [ID, "OOD_AP_ADULT"]:
            e_ = est(rid, c, "risk_change")
            want = (e_.ci_high < 0) if c == ID else (e_.ci_low > 0)
            checks.append(dict(run=rid, conclusion=f"referral {'reduces' if c == ID else 'increases'} error on {c}", value=e_.estimate,
                               ci_low=e_.ci_low, ci_high=e_.ci_high, holds=bool(want)))
        e_ = est(rid, "OOD_AP_ADULT", "spec_A")
        checks.append(dict(run=rid, conclusion="NIH AP specificity < 0.20", value=e_.estimate, ci_low=e_.ci_low, ci_high=e_.ci_high, holds=bool(e_.ci_high < 0.20)))
CHK = pd.DataFrame(checks); CHK.to_csv(DIRS["tables"] / "R1_conclusion_checks.csv", index=False)
print(CHK.pivot_table(index="conclusion", columns="run", values="holds", aggfunc="first").to_string())
print("\nAll primary conclusions seed-robust:", bool(CHK.holds.all()))
# architecture difference on NIH AP across seeds (ViT − B4, same seed)
arch = []
for s in [42] + [sd for p_, sd in NEW_RUNS if p_ == "lb"]:
    for c in EVAL:
        dd = BOOT[(run_id("vit_b16", "lb", s), c)].auroc.values - BOOT[(run_id("efficientnet_b4", "lb", s), c)].auroc.values
        arch.append(dict(seed=s, cohort=c, vit_minus_b4=est(run_id("vit_b16", "lb", s), c, "auroc").estimate - est(run_id("efficientnet_b4", "lb", s), c, "auroc").estimate,
                         ci_low=np.nanpercentile(dd, 2.5), ci_high=np.nanpercentile(dd, 97.5)))
pd.DataFrame(arch).to_csv(DIRS["tables"] / "R1_architecture_difference_by_seed.csv", index=False)
print(pd.DataFrame(arch).round(3).to_string(index=False))

## 8. R2 — letterbox padding: stretch − letterbox (seed 42), paired bootstrap

In [ ]:
pad = []
for m in MODELS:
    a, b = run_id(m, "lb", 42), run_id(m, "st", 42)
    for c in EVAL:
        for k in ["auroc", "spec_A", "sens_A", "ece_ts", "errdet_auroc", "risk_change", "g4_pct_of_errors"]:
            dd = BOOT[(b, c)][k].values - BOOT[(a, c)][k].values
            pad.append(dict(model=m, cohort=c, metric=k, letterbox=est(a, c, k).estimate, stretch=est(b, c, k).estimate,
                            difference=est(b, c, k).estimate - est(a, c, k).estimate, ci_low=np.nanpercentile(dd, 2.5), ci_high=np.nanpercentile(dd, 97.5)))
PAD = pd.DataFrame(pad); PAD.to_csv(DIRS["tables"] / "R2_padding_ablation.csv", index=False)
print(PAD[PAD.metric.isin(["auroc", "spec_A", "errdet_auroc"])].round(3).to_string(index=False))

# false-positive rate among ID-test normals by aspect-ratio quartile (edges from ID-test normals; identical for all runs)
nI = COH[ID][COH[ID].y == 0]; edges = np.quantile(nI.aspect, [0, .25, .5, .75, 1]); edges[-1] += 1e-9
q_rows = []
for rid in RUNS:
    d = G[(rid, ID)]; normal = d.y.values == 0; qd = np.clip(np.searchsorted(edges, d.aspect.values, side="right") - 1, 0, 3)
    codes, dr = DRAWS[ID]
    def fpq(w):
        return [((w * normal * (qd == k) * d.pred.values).sum() / max((w * normal * (qd == k)).sum(), 1e-12)) for k in range(4)]
    pt = fpq(np.ones(len(d))); bs = np.array([fpq(cnt[codes].astype(float)) for cnt in dr])
    for k in range(4):
        q_rows.append(dict(run=rid, quartile=f"Q{k + 1}", fp_rate=pt[k], ci_low=np.percentile(bs[:, k], 2.5), ci_high=np.percentile(bs[:, k], 97.5)))
    q_rows.append(dict(run=rid, quartile="Q4 − Q1", fp_rate=pt[3] - pt[0], ci_low=np.percentile(bs[:, 3] - bs[:, 0], 2.5), ci_high=np.percentile(bs[:, 3] - bs[:, 0], 97.5)))
QA = pd.DataFrame(q_rows); QA.to_csv(DIRS["tables"] / "R2_fp_rate_by_aspect_quartile.csv", index=False)
print(QA[QA.quartile == "Q4 − Q1"].round(3).to_string(index=False))

## 9. Overview figure and output ZIP

In [ ]:
SHORT = {ID: "ID test", "OOD_AP_ADULT": "NIH AP", "OOD_PA_ADULT": "NIH PA", "OOD_HARD_AP_ADULT": "NIH AP-Hard"}
COL = {"efficientnet_b4": "#2a78d6", "vit_b16": "#eb6834"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, k, lab in zip(axs, ["auroc", "errdet_auroc", "risk_change"], ["AUROC", "Error-detection AUROC (MC entropy)", "Selective risk − error rate (τ90)"]):
    for j, m in enumerate(MODELS):
        for rid, o in OPS.items():
            if o["model"] != m: continue
            xs = np.arange(len(EVAL)) + (j - 0.5) * 0.3 + (0.04 * (o["seed"] - 43) if o["pre"] == "lb" else 0.08)
            e_ = R[(R.run == rid) & (R.metric == k)].set_index("cohort").loc[EVAL]
            ax.errorbar(xs, e_.estimate, yerr=[e_.estimate - e_.ci_low, e_.ci_high - e_.estimate], fmt="o" if o["pre"] == "lb" else "^",
                        ms=4, lw=0.8, color=COL[m], mfc=COL[m] if o["pre"] == "lb" else "white",
                        label=f"{m.replace('efficientnet_b4', 'EfficientNet-B4').replace('vit_b16', 'ViT-B/16')} {'letterbox s' + str(o['seed']) if o['pre'] == 'lb' else 'stretch s42'}")
    ax.axhline(0.5 if k != "risk_change" else 0, color="#888888", lw=0.7, ls=":")
    ax.set_xticks(range(len(EVAL))); ax.set_xticklabels([SHORT[c] for c in EVAL]); ax.set_ylabel(lab)
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=4, frameon=False, fontsize=7)
fig.tight_layout(rect=(0, 0.12, 1, 1))
for ext in ["png", "pdf"]: fig.savefig(DIRS["figures"] / f"figure_robustness_overview.{ext}", dpi=300)
plt.close(fig)

summary = dict(runs=OPS, n_boot=N_BOOT, T_MC=T_MC, all_conclusions_seed_robust=bool(CHK.holds.all()), env=ENV, smoke_test=SMOKE_TEST)
(DIRS["metrics"] / "notebook06_summary.json").write_text(json.dumps(summary, indent=2, default=str))
zb = PROJECT / "phase6_robustness_outputs"; tmp = PROJECT / "_zip06"; shutil.rmtree(tmp, ignore_errors=True)
for k in ["predictions", "metrics", "tables", "figures", "logs"]: shutil.copytree(DIRS[k], tmp / k, dirs_exist_ok=True)
shutil.make_archive(str(zb), "zip", tmp); shutil.rmtree(tmp)
print("ZIP →", f"{zb}.zip", f"({os.path.getsize(f'{zb}.zip') / 1e6:.0f} MB)")
print("Notebook 06 complete. Download phase6_robustness_outputs.zip and upload it to Claude.")